In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import math
import keras
import torch
import tensorflow as tf
from keras.utils import image_dataset_from_directory as loader

### System details

In [2]:
print(f"PyTorch version: {torch.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"Tensorflow version: {tf.__version__}")

print("--------------------------------------------------")
print(f"Using cuda: {torch.cuda.is_available()}")
print(f"Cuda device: {torch.cuda.get_device_name(torch.cuda.current_device())}")

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(f"Device Name: {tf.test.gpu_device_name()}")

PyTorch version: 2.2.0+cu118
Keras version: 3.0.5
Tensorflow version: 2.10.1
--------------------------------------------------
Using cuda: True
Cuda device: NVIDIA GeForce GTX 1660 SUPER
Num GPUs Available:  1
Device Name: /device:GPU:0


### Getting the data

In [3]:
root_dir = "../PKLot/PKLotSegmented"

ds_train = []
ds_test = []
ds_validation = []

total_days, train_days, test_days, validation_days = (0, 0, 0, 0)
temp = 0

subsets = os.listdir(root_dir)
for subset in subsets:
    climatic_condition = os.listdir(f"{root_dir}/{subset}")
    for weather in climatic_condition:
        dates = os.listdir(f"{root_dir}/{subset}/{weather}")

        total_days += len(dates)

        days = len(dates) - 1
        test_qtd = math.floor(days * .5)
        train_qtd = days - test_qtd

        count = len(dates)
        for index, date in enumerate(dates):
            if count == len(dates):
                ds_validation.append(loader(f"{root_dir}/{subset}/{weather}/{date}", image_size=(224, 224)))
                validation_days += 1
            elif count > test_qtd:
                ds_train.append(loader(f"{root_dir}/{subset}/{weather}/{date}", image_size=(224, 224)))
                train_days += 1
            else:
                ds_test.append(loader(f"{root_dir}/{subset}/{weather}/{date}", image_size=(224, 224)))
                test_days += 1
            
            count -= 1

Found 5098 files belonging to 2 classes.
Found 13197 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
Found 15400 files belonging to 2 classes.
Found 15498 files belonging to 2 classes.
Found 14900 files belonging to 2 classes.
Found 14496 files belonging to 2 classes.
Found 15600 files belonging to 2 classes.
Found 13498 files belonging to 2 classes.
Found 16193 files belonging to 2 classes.
Found 7900 files belonging to 2 classes.
Found 2400 files belonging to 2 classes.
Found 9496 files belonging to 2 classes.
Found 18488 files belonging to 2 classes.
Found 1100 files belonging to 2 classes.
Found 13597 files belonging to 2 classes.
Found 15891 files belonging to 2 classes.
Found 16091 files belonging to 2 classes.
Found 5993 files belonging to 2 classes.
Found 3571 files belonging to 2 classes.
Found 8999 files belonging to 2 classes.
Found 12399 files belonging to 2 classes.
Found 11076 files belonging to 2 classes.
Found 14792 files belonging to 2 classes.
F

In [4]:
final_train = ds_train[0].concatenate(ds_train[1])
for index in range(2, len(ds_train)):
    final_train = final_train.concatenate(ds_train[index])

final_test = ds_test[0].concatenate(ds_test[1])
for index in range(2, len(ds_test)):
    final_test = final_test.concatenate(ds_test[index])

final_validation = ds_validation[0].concatenate(ds_test[1])
for index in range(2, len(ds_validation)):
    final_validation = final_validation.concatenate(ds_validation[index])

In [5]:
print(f"Dataset days: {total_days}")
print(f"Training days: {train_days} - {final_train}")
print(f"Testing days: {test_days} - {final_test}")
print(f"Validation days: {validation_days} - {final_validation}")

Dataset days: 145
Training days: 70 - <ConcatenateDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>
Testing days: 66 - <ConcatenateDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>
Validation days: 9 - <ConcatenateDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>


### Transfer learning

In [6]:
base_model = keras.applications.MobileNetV3Large(weights="imagenet", include_top=False)
base_model.summary()

c:\Python310\lib\site-packages\keras\src\applications\mobilenet_v3.py:512: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


Model: "MobilenetV3large"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None,      │          0 │ -                 │
│ (InputLayer)        │ None, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, None,      │          0 │ input_layer[0][0] │
│ (Rescaling)         │ None, 3)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv (Conv2D)       │ (None, None,      │        432 │ rescaling[0][0]   │
│                     │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_bn             │ (None, None,      │         64 │ conv[0][0]        │
│ (BatchNormalizatio… │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, None,      │          0 │ conv_bn[0][0]     │
│ (Activation)        │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, None,      │        144 │ activation[0][0]  │
│ (DepthwiseConv2D)   │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, None,      │         64 │ expanded_conv_de… │
│ (BatchNormalizatio… │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, None,      │          0 │ expanded_conv_de… │
│                     │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, None,      │        256 │ re_lu[0][0]       │
│ (Conv2D)            │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, None,      │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ None, 16)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_add   │ (None, None,      │          0 │ activation[0][0], │
│ (Add)               │ None, 16)         │            │ expanded_conv_pr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_ex… │ (None, None,      │      1,024 │ expanded_conv_ad… │
│ (Conv2D)            │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_ex… │ (None, None,      │        256 │ expanded_conv_1_… │
│ (BatchNormalizatio… │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, None,      │          0 │ expanded_conv_1_… │
│                     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, None,      │          0 │ re_lu_1[0][0]     │
│ (ZeroPadding2D)     │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, None,      │        576 │ expanded_conv_1_… │
│ (DepthwiseConv2D)   │ None, 64)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, None,      │        256 │ expanded_conv_1_

 Total params: 2,996,352 (11.43 MB)

 Trainable params: 2,971,952 (11.34 MB)

 Non-trainable params: 24,400 (95.31 KB)

In [7]:
base_model.trainable = False
keras_input = keras.Input(shape=(224, 224, 3))

new_layer = base_model(keras_input, training=False)
new_layer = keras.layers.GlobalAveragePooling2D()(new_layer)
outputs = keras.layers.Dense(1)(new_layer)
model = keras.Model(keras_input, outputs)

model.summary(show_trainable=True)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer_1 (InputLayer)  │ (None, 224, 224, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ MobilenetV3large            │ (None, 7, 7, 960)     │  2,996,352 │   N   │
│ (Functional)                │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ global_average_pooling2d    │ (None, 960)           │          0 │   -   │
│ (GlobalAveragePooling2D)    │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense (Dense)               │ (None, 1)             │        961 │   Y   │
└─────────────────────────────┴───────────────────────┴────────────┴───────┘

 Total params: 2,997,313 (11.43 MB)

 Trainable params: 961 (3.75 KB)

 Non-trainable params: 2,996,352 (11.43 MB)

In [8]:
model.compile(optimizer=keras.optimizers.Adam(), loss=keras.losses.BinaryCrossentropy(from_logits=True), metrics=[keras.metrics.BinaryAccuracy()])

print("fitting only new layer")
model.fit(final_train, epochs=2, validation_data=final_validation)

fitting only new layer
Epoch 1/2
11219/11219 ━━━━━━━━━━━━━━━━━━━━ 1100s 98ms/step - binary_accuracy: 0.9982 - loss: 0.0115 - val_binary_accuracy: 0.9961 - val_loss: 0.0158
Epoch 2/2
11219/11219 ━━━━━━━━━━━━━━━━━━━━ 1013s 90ms/step - binary_accuracy: 0.9987 - loss: 0.0070 - val_binary_accuracy: 0.9962 - val_loss: 0.0148


### Fine tuning

In [9]:
base_model.trainable = True
model.summary(show_trainable=True)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer_1 (InputLayer)  │ (None, 224, 224, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ MobilenetV3large            │ (None, 7, 7, 960)     │  2,996,352 │   Y   │
│ (Functional)                │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ global_average_pooling2d    │ (None, 960)           │          0 │   -   │
│ (GlobalAveragePooling2D)    │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense (Dense)               │ (None, 1)             │        961 │   Y   │
└─────────────────────────────┴───────────────────────┴────────────┴───────┘

 Total params: 2,999,237 (11.44 MB)

 Trainable params: 2,972,913 (11.34 MB)

 Non-trainable params: 24,400 (95.31 KB)

 Optimizer params: 1,924 (7.52 KB)

In [10]:
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss=keras.losses.BinaryCrossentropy(from_logits=True), metrics=[keras.metrics.BinaryAccuracy()])

print("Fitting hole model")
model.fit(final_train, epochs=1, validation_data=final_validation)

Fitting hole model
11219/11219 ━━━━━━━━━━━━━━━━━━━━ 3739s 333ms/step - binary_accuracy: 0.9912 - loss: 0.0360 - val_binary_accuracy: 0.9971 - val_loss: 0.0105


### Tests and results

In [11]:
model.evaluate(final_test)

9983/9983 ━━━━━━━━━━━━━━━━━━━━ 882s 88ms/step - binary_accuracy: 0.9964 - loss: 0.0187


[0.026643572375178337, 0.9946431517601013]